In [1]:
import os
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()

url = os.getenv("SUPABASE_URL2")
key = os.getenv("SUPABASE_ANON_KEY2")

supabase = create_client(url, key)

print(url)

https://narojnxvhizepvyhywsu.supabase.co


In [7]:
# 로그인
response = supabase.auth.sign_in_with_password({
    "email":"test4@test.org",
    "password":"password1234"
})


# 현재 로그인된 사용자 확인
user = supabase.auth.get_user() # 현재 로그인한 사용자 정보, 있으면 로그인 상태, none이면 미인증상태
user

UserResponse(user=User(id='1deda96d-cd0c-4496-a048-b076873e3566', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'test4@test.org', 'email_verified': True, 'phone_verified': False, 'sub': '1deda96d-cd0c-4496-a048-b076873e3566'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='test4@test.org', phone='', created_at=datetime.datetime(2026, 9, 10, 12, 39, 18, 321242, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 10, 12, 39, 18, 362166, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 10, 12, 39, 18, 362166, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 10, 13, 3, 35, 170954, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 10, 13, 3, 35, 183375, tzinfo=TzInfo(0)), identities=[UserIdentity(id='1deda96d-cd0c-4496-a048-b076873e3566', ide

In [8]:
class product_service:
    def __init__(self):
        # 현재 로그인되어 있는 사용자 정보를 Supabase에서 가져온다.
        data = supabase.auth.get_user()

        if data:
            # data.user = 현재 로그인한 사용자
            # self.user에 저장해두면 다른 메서드에서도 계속 사용할 수 있다.
            self.user = data.user

    def create(self, name, price=0, description=None):
        # 상품 생성 권한 확인
        # True = "지금은 상품 생성 작업이다"
        # → 판매자인지만 검사하고, 기존 상품 주인인지는 검사하지 않는다.
        self.check_authority(True)

        response = (
            supabase.table("products")       # products 테이블에
            .insert({
                "name": name,                # name 컬럼에 전달받은 name
                "price": price,              # price 컬럼에 전달받은 price
                "description": description,  # description 컬럼에 전달받은 description
                # 현재 로그인한 사용자의 id를 seller_id로 저장
                "seller_id": self.user.id
            })
            .execute()                       # 실제 INSERT 실행
        )

        # Supabase가 돌려준 결과 중 데이터만 반환
        return response.data

    def update(self, id, name=None, price=None, description=None):
        # 수정할 컬럼들만 담을 빈 딕셔너리
        updated_data = {}

        # id에 해당하는 상품을 먼저 조회
        product = self.get_product(id)

        # 이 상품의 seller_id와 현재 로그인 사용자의 id를 비교하여
        # 수정 권한이 있는지 확인
        self.check_authority(user_id=product["seller_id"])

        # name을 전달한 경우에만 name 수정
        if name is not None:
            updated_data["name"] = name

        # price를 전달한 경우에만 price 수정
        if price is not None:
            updated_data["price"] = price

        # description을 전달한 경우에만 description 수정
        if description is not None:
            updated_data["description"] = description

        response = (
            supabase.table("products")       # products 테이블에서
            .update(updated_data)             # updated_data에 들어있는 컬럼만 수정
            .eq("id", id)                    
            .execute()
        )

        return response.data

    def delete(self, id):
        # 삭제하려는 상품 조회
        product = self.get_product(id)

        # 이 상품을 등록한 판매자가 현재 로그인 사용자와 같은지 확인
        self.check_authority(user_id=product["seller_id"])

        response = (
            supabase.table("products")       # products 테이블에서
            .delete()                         # DELETE
            .eq("id", id)                     # id가 같은 상품 하나만
            .execute()
        )
        return response.data

    def get_product(self, id):
        # 특정 상품 1개 조회
        response = (
            supabase.table("products")       # products 테이블에서
            .select("*")                      # 모든 컬럼 조회
            .eq("id", id)                     # id가 전달받은 id와 같은 상품만
            .execute()
        )

        # response.data는 보통 리스트 형태
        # 예: [{"id": "...", "name": "apple", ...}]
        # 상품 하나만 필요하므로 [0]으로 첫 번째 상품 반환
        return response.data[0]

    def get_products(self, page=1, limit=10, *, search=None):
        # products 테이블 전체 조회 쿼리 만들기
        query = (
            supabase.table("products")
            .select("*")
        )

        # 검색어가 있다면
        if search:
            # name에 search 문자열이 포함된 상품만 검색
            # 예: search="apple"
            # → %apple% = 앞뒤에 어떤 글자가 있어도 apple 포함
            query = query.ilike("name", f"%{search}%")

        # page와 limit을 이용해서 시작 위치 계산
        start = (page - 1) * limit

        # 마지막 위치 계산
        end = start + limit - 1

        response = (
            query
            .order("id")                      # id 기준으로 정렬
            .range(start, end)                 # 해당 페이지에 필요한 상품만 가져옴
            .execute()
        )
        return response.data

    def is_seller(self):
        # 로그인한 사용자가 없으면 판매자 아님
        if not self.user:
            return False

        # user_details에서 현재 사용자의 type 조회
        response = (
            supabase.table("user_details")
            .select("type")
            .eq("id", self.user.id)
            .limit(1)
            .execute()
        )

        # user_details에 사용자 정보가 없으면 판매자 아님
        if not response.data:
            return False

        # type이 SELLER인지 확인
        return response.data[0]["type"] == "SELLER"


    def check_authority(self, is_created=False, *, user_id=None):
        # 현재 사용자가 판매자가 아니라면 바로 에러
        if not self.is_seller():
            raise PermissionError("판매자만 처리할 수 있습니다.")

        # create가 아닌 경우(update/delete)
        # 상품을 등록한 seller_id와 현재 로그인 사용자 id를 비교
        if (not is_created) and user_id != self.user.id:
            raise PermissionError(
                "상품을 등록한 판매자만 수정 또는 삭제할 수 있습니다."
            )

In [9]:
new_product = product_service()

new_product.create("촉촉한 황치즈칩", "4480", "오리온 촉촉한 황치즈칩 16개입(320g)")

[{'id': '1ed9837b-f9cf-4546-8859-7c3626273f5e',
  'name': '촉촉한 황치즈칩',
  'description': '오리온 촉촉한 황치즈칩 16개입(320g)',
  'price': 4480,
  'seller_id': '1deda96d-cd0c-4496-a048-b076873e3566',
  'created_at': '2026-09-10T13:08:31.785899+00:00',
  'modified_at': '2026-09-10T13:08:31.785899+00:00',
  'deleted_at': None}]

In [17]:
created_product = new_product.create("미쯔 황치즈맛", "990", "오리온 미쯔 황치즈맛")

In [18]:
product_id = created_product[0]["id"]
new_product.update(id=product_id, name="미쯔 황치즈맛 NEW")

[{'id': 'cceeac13-7164-4d63-b73c-a54095d92342',
  'name': '미쯔 황치즈맛 NEW',
  'description': '오리온 미쯔 황치즈맛',
  'price': 990,
  'seller_id': '1deda96d-cd0c-4496-a048-b076873e3566',
  'created_at': '2026-09-10T13:15:57.478494+00:00',
  'modified_at': '2026-09-10T13:15:57.478494+00:00',
  'deleted_at': None}]

In [19]:
new_product.delete(id=product_id)

[{'id': 'cceeac13-7164-4d63-b73c-a54095d92342',
  'name': '미쯔 황치즈맛 NEW',
  'description': '오리온 미쯔 황치즈맛',
  'price': 990,
  'seller_id': '1deda96d-cd0c-4496-a048-b076873e3566',
  'created_at': '2026-09-10T13:15:57.478494+00:00',
  'modified_at': '2026-09-10T13:15:57.478494+00:00',
  'deleted_at': None}]